# Master Model Comparison & Benchmarking

**Project:** Encrypted DNS & QUIC Traffic Intelligence  
**Focus:** Comprehensive Multi-Model Benchmarking Across Observable Kinematic Features  
**Models Evaluated:**
1. Random Forest (`RandomForestClassifier`)
2. XGBoost (`XGBClassifier`)
3. Decision Tree (`DecisionTreeClassifier`)
4. Support Vector Machine (`SVC`)
5. k-Nearest Neighbors (`KNeighborsClassifier`)
6. Logistic Regression (`LogisticRegression`)
7. Multi-Layer Perceptron (`MLPClassifier`)

---

### Objective
Provide an end-to-end head-to-head comparison on the exact same train/validation/test partitions, evaluating:
- Test Accuracy & Macro-F1
- Per-Protocol Precision & Recall
- Inference Latency per Packet Vector
- Within-Network vs. Cross-Network Generalization


In [ ]:
# ── 1. Setup & Imports ──
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Helvetica'

RANDOM_STATE = 42
print("Benchmark dependencies initialized.")


In [ ]:
# ── 2. Ingest Consolidated Dataset ──
DATA_DIR = Path("../PROJECT_DATASET")
DATASET_PATH = DATA_DIR / "FULL_COMBINED_WITH_METADATA.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = DATA_DIR / "FULL_COMBINED.csv"

df = pd.read_csv(DATASET_PATH)
FEATURE_COLS = ['Current packet size', 'Previous packet size', 'Current time gap', 'Previous time gap']
CLASS_MAP = {'DOH': 0, 'DOQ': 1, 'DOH3': 2, 'QUIC': 3, 'HTTPS_WEB': 4, 'HTTP3_WEB': 5}
CLASS_NAMES = {v: k for k, v in CLASS_MAP.items()}

X = df[FEATURE_COLS].values
y = df['traffic_class'].map(CLASS_MAP).values

# 70 / 15 / 15 Stratified Split
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=(0.15 / 0.85), stratify=y_train_val, random_state=RANDOM_STATE)

print(f"Dataset: {len(df):,d} samples.")
print(f"Train: {len(X_train):,d} | Val: {len(X_val):,d} | Test: {len(X_test):,d}")


In [ ]:
# ── 3. Configure All 7 Candidate Models ──
models_dict = {
    'Random Forest': RandomForestClassifier(n_estimators=200, criterion='entropy', random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(criterion='entropy', max_depth=12, random_state=RANDOM_STATE),
    'SVM (RBF)': Pipeline([('scaler', StandardScaler()), ('clf', SVC(C=10.0, kernel='rbf', probability=True, random_state=RANDOM_STATE))]),
    'KNN': Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=7, weights='distance', n_jobs=-1))]),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE))]),
    'MLP Neural Net': Pipeline([('scaler', StandardScaler()), ('clf', MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200, random_state=RANDOM_STATE))]),
}

results = []
print("Starting head-to-head benchmarking on test partition...")

for name, model in models_dict.items():
    print(f"Training {name:20s}...", end="")
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - t0

    t_inf = time.time()
    y_pred = model.predict(X_test)
    inf_latency = (time.time() - t_inf) / len(X_test) * 1000

    acc = accuracy_score(y_test, y_pred)
    f1_m = f1_score(y_test, y_pred, average='macro')
    f1_w = f1_score(y_test, y_pred, average='weighted')
    prec_m = precision_score(y_test, y_pred, average='macro')
    rec_m = recall_score(y_test, y_pred, average='macro')

    results.append({
        'Model': name,
        'Accuracy': acc,
        'Macro-F1': f1_m,
        'Weighted-F1': f1_w,
        'Precision': prec_m,
        'Recall': rec_m,
        'Fit Time (s)': fit_time,
        'Inference (ms/pkt)': inf_latency
    })
    print(f" Done! Acc: {acc:.4f} | Macro-F1: {f1_m:.4f} ({fit_time:.2f}s)")

df_results = pd.DataFrame(results).sort_values('Macro-F1', ascending=False).reset_index(drop=True)


In [ ]:
# ── 4. Leaderboard & Visual Comparison ──
print("\n==========================================================================")
print("                       OFFICIAL MODEL LEADERBOARD")
print("==========================================================================")
print(df_results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Macro-F1 vs Accuracy
sns.barplot(x='Macro-F1', y='Model', data=df_results, ax=axes[0], palette='Blues_r')
axes[0].set_title('Macro-F1 Score by Model', fontweight='bold')
axes[0].set_xlim(0.8, 1.01)

# Plot 2: Inference Latency
sns.barplot(x='Inference (ms/pkt)', y='Model', data=df_results, ax=axes[1], palette='Reds_r')
axes[1].set_title('Inference Latency (ms per packet vector)', fontweight='bold')
plt.tight_layout()
plt.show()
